# Lab 8. Data Wrangling: Join, Combine and Reshape


## What you will do

| Part | Topic | Tasks | Points |
|---|---|---|---|
| 1 | Hierarchical indexing | 4 | 12 |
| 2 | `merge` and `join` | 6 | 26 |
| 3 | `concat` | 3 | 14 |
| 4 | `combine_first` | 1 | 8 |
| 5 | `pivot`, `melt`, `stack`, `unstack` | 4 | 22 |
| 6 | Mini project: revenue report | 3 | 18 |
| | **Total** | **21** | **100** |

## Rules

1. Go **top to bottom**. Replace every `None  # YOUR CODE HERE` with your code.
2. Do not edit the *given data* cells or the *check* cells.
3. Every task ends with a **check cell** that prints `PASS` or `FAIL`. It never stops the notebook.
4. The last cell shows your automatic score.
5. Before you submit: **Kernel > Restart & Run All**, make sure nothing raises an error, save, upload the `.ipynb`.
6. **Defense:** at the end there are questions you will answer aloud. You must be able to explain every line of your own code.

## Setup
Run this cell first. It imports the libraries and defines the self-check helper.

In [2]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__)

# ----- Self-check helper. Do not edit this cell. -----
POINTS = {"1.1": 3, "1.2": 3, "1.3": 3, "1.4": 3, "2.1": 3, "2.2": 4, "2.3": 5, "2.4": 4, "2.5": 6, "2.6": 4, "3.1": 5, "3.2": 4, "3.3": 5, "4.1": 8, "5.1": 6, "5.2": 6, "5.3": 5, "5.4": 5, "6.1": 7, "6.2": 6, "6.3": 5}
_results = {}


def check(task_id, condition):
    """Run `condition` (a lambda returning True/False), print PASS or FAIL and remember the result."""
    try:
        ok = bool(condition())
        print(f"Task {task_id}: {'PASS' if ok else 'FAIL'}")
    except Exception as exc:
        ok = False
        print(f"Task {task_id}: FAIL ({type(exc).__name__}: {exc})")
    _results[task_id] = ok


def summary():
    """Print the automatic score per task and per part."""
    rows, by_part, part_max = [], {}, {}
    for tid, pts in POINTS.items():
        got = pts if _results.get(tid) else 0
        part = tid.split(".")[0]
        by_part[part] = by_part.get(part, 0) + got
        part_max[part] = part_max.get(part, 0) + pts
        rows.append((tid, pts, got, "PASS" if _results.get(tid) else ("FAIL" if tid in _results else "not run")))
    display(pd.DataFrame(rows, columns=["task", "max", "earned", "result"]).set_index("task"))
    print()
    for part in sorted(part_max, key=int):
        print(f"Part {part}: {by_part[part]:>3} / {part_max[part]}")
    print(f"TOTAL (automatic): {sum(by_part.values())} / {sum(part_max.values())}")

pandas 3.0.6 | numpy 2.5.3


## Given data: a small online store

| Table | Key | Meaning |
|---|---|---|
| `customers` | `customer_id` | who buys |
| `products` | `product_id` | what is sold |
| `orders` | `order_id`, `customer_id` | one row per order |
| `order_items` | `order_id`, `product_id` | one row per product line of an order |

Traps in the data: order `1009` belongs to customer `99` who does not exist, customers `7` and `8` never ordered,
order `1004` is cancelled, and both `orders` and `customers` have a column named `status`.

In [3]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "name": ["Aigerim", "Bekzat", "Dana", "Erlan", "Firuza", "Gulnara", "Hasan", "Ilyas"],
    "city": ["Almaty", "Astana", "Almaty", "Shymkent", "Astana", "Karaganda", "Almaty", "Shymkent"],
    "status": ["gold", "standard", "gold", "standard", "standard", "gold", "standard", "standard"],
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105, 106],
    "product_name": ["Laptop", "Headphones", "Notebook", "Pen", "Backpack", "Desk Lamp"],
    "category": ["Electronics", "Electronics", "Stationery", "Stationery", "Accessories", "Home"],
    "price": [450000, 25000, 800, 150, 12000, 9000],
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012],
    "customer_id": [1, 2, 1, 3, 4, 2, 5, 3, 99, 6, 1, 5],
    "order_date": ["2026-01-05", "2026-01-09", "2026-02-03", "2026-02-11", "2026-02-20", "2026-03-02",
                   "2026-03-15", "2026-03-18", "2026-03-21", "2026-04-04", "2026-04-12", "2026-04-25"],
    "status": ["paid", "paid", "paid", "cancelled", "paid", "paid", "paid", "paid", "paid", "paid", "paid", "paid"],
})
orders["order_date"] = pd.to_datetime(orders["order_date"])

order_items = pd.DataFrame({
    "order_id":   [1001, 1001, 1002, 1002, 1003, 1004, 1005, 1006, 1006, 1007, 1008, 1009, 1010, 1011, 1012],
    "product_id": [101,  102,  103,  104,  105,  102,  101,  102,  103,  105,  104,  101,  103,  102,  105],
    "quantity":   [1,    2,    10,   20,   1,    1,    1,    1,    5,    2,    50,   1,    8,    3,    1],
})

for name, df in [("customers", customers), ("products", products), ("orders", orders), ("order_items", order_items)]:
    print(f"{name:12s} {df.shape}")
orders.head()

customers    (8, 4)
products     (6, 4)
orders       (12, 4)
order_items  (15, 3)


,order_id,customer_id,order_date,status
0,1001,1,2026-01-05,paid
1,1002,2,2026-01-09,paid
2,1003,1,2026-02-03,paid
3,1004,3,2026-02-11,cancelled
4,1005,4,2026-02-20,paid


---
# Part 1. Hierarchical indexing (12 points)

A `MultiIndex` gives one axis several levels of labels.

| Goal | Tool |
|---|---|
| select an outer level | `s["North"]` |
| select an inner level | `s.loc[:, "Q1"]` |
| inner level to columns | `s.unstack()` |
| columns to index and back | `df.set_index([...])` / `df.reset_index()` |

In [4]:
index = pd.MultiIndex.from_tuples(
    [("North", "Q1"), ("North", "Q2"), ("North", "Q3"),
     ("South", "Q1"), ("South", "Q3"),
     ("East", "Q1"), ("East", "Q2"), ("East", "Q4")],
    names=["region", "quarter"],
)
rev = pd.Series([120, 135, 150, 90, 110, 70, 80, 95], index=index, name="revenue")   # thousand KZT
rev

region  quarter
North   Q1         120
        Q2         135
        Q3         150
South   Q1          90
        Q3         110
East    Q1          70
        Q2          80
        Q4          95
Name: revenue, dtype: int64

### Task 1.1 (3 pts): Select an outer level

Select **all quarters of the North region** from `rev` into `rev_north`.

In [11]:
rev_north = rev.loc["North"]  # YOUR CODE HERE
rev_north

quarter
Q1    120
Q2    135
Q3    150
Name: revenue, dtype: int64

In [12]:
check("1.1", lambda: rev_north.sum() == 405 and list(rev_north.index) == ["Q1", "Q2", "Q3"])

Task 1.1: PASS


### Task 1.2 (3 pts): Select an inner level

Select the **Q1 revenue of every region** into `q1` (a Series indexed by region).

In [15]:
q1 = rev.loc[:, "Q1"]  # YOUR CODE HERE
q1

region
North    120
South     90
East      70
Name: revenue, dtype: int64

In [14]:
check("1.2", lambda: q1.to_dict() == {"North": 120, "South": 90, "East": 70})

Task 1.2: PASS


### Task 1.3 (3 pts): unstack

Make a table `rev_table` with **regions as rows and quarters as columns**. Missing combinations must be `0`.

In [23]:
rev_table = rev.unstack(level='quarter', fill_value=0)  # YOUR CODE HERE
rev_table

quarter,Q1,Q2,Q3,Q4
region,,,,
East,70,80,0,95
North,120,135,150,0
South,90,0,110,0


In [24]:
check("1.3", lambda: rev_table.shape == (3, 4) and rev_table.loc["South", "Q2"] == 0 and rev_table.loc["East", "Q4"] == 95)

Task 1.3: PASS


### Task 1.4 (3 pts): set_index and reset_index

Using `customers`:

1. `by_city`: make `city` and `customer_id` the row index and sort the index.
2. `almaty_customers`: select the customers of Almaty from `by_city`.
3. `flat`: turn the index levels of `by_city` back into ordinary columns.

In [25]:
# set_index with a LIST of columns builds a MultiIndex (level order = list order);
# sort_index() orders rows by city first, then customer_id. Sorted MultiIndexes make .loc slicing fast and safe.
by_city = customers.set_index(["city", "customer_id"]).sort_index()

# .loc with a single label selects on the OUTER level (city) and drops that level from the result.
almaty_customers = by_city.loc["Almaty"]

# reset_index() moves every index level back into ordinary columns (index levels first, then old columns),
# giving exactly ["city", "customer_id", "name", "status"], and restores a plain 0..n-1 index.
flat = by_city.reset_index()

In [26]:
check("1.4", lambda: list(by_city.index.names) == ["city", "customer_id"] and len(almaty_customers) == 3
      and list(flat.columns) == ["city", "customer_id", "name", "status"])

Task 1.4: PASS


---
# Part 2. Merge and join (26 points)

`pd.merge(left, right, how=..., on=...)` works like a SQL join.

| `how` | Rows kept |
|---|---|
| `inner` (default) | keys in **both** tables |
| `left` | all keys of the left table |
| `right` | all keys of the right table |
| `outer` | all keys of both tables |

### Task 2.1 (3 pts): predict first
`orders` has 12 rows and `customers` has 8. **Without running anything**, predict how many rows a merge on `customer_id`
returns for each `how`.

In [29]:
# Reasoning: orders has 12 rows; only order 1009 has an unknown customer (99). Customers 7 and 8 never ordered.
# inner : keys in BOTH tables -> 12 - 1 (order 1009) = 11
# left  : all orders (left table) -> 12 (1009 kept, customer columns NaN)
# right : the 11 matched orders + customers 7 and 8 with no orders (one row each) -> 13
# outer : the union of everything -> 11 matched + 1 orphan order + 2 idle customers = 14
predicted = {"inner": 11, "left": 12, "right": 13, "outer": 14}

### Tasks 2.1 and 2.2 (3 + 4 pts): count the rows

Store in `actual_counts` the number of rows of the four merges of `orders` with `customers` on `customer_id`
(keys `"inner"`, `"left"`, `"right"`, `"outer"`). The check also grades your prediction from 2.1.

*Hint:* a dict comprehension over the four `how` values.

In [27]:
# One merge per join type; len() counts the resulting rows.
actual_counts = {how: len(pd.merge(orders, customers, on="customer_id", how=how))
                 for how in ["inner", "left", "right", "outer"]}

In [30]:
print("predicted:", predicted)
print("actual:   ", actual_counts)
check("2.1", lambda: predicted == actual_counts)
check("2.2", lambda: actual_counts == {"inner": 11, "left": 12, "right": 13, "outer": 14})

predicted: {'inner': 11, 'left': 12, 'right': 13, 'outer': 14}
actual:    {'inner': 11, 'left': 12, 'right': 13, 'outer': 14}
Task 2.1: PASS
Task 2.2: PASS


### Task 2.3 (5 pts): Many-to-one merge

Add product information to every line of `order_items`: merge `order_items` with `products` on `product_id`
into `items_products`. All 15 lines must survive.

In [31]:
# Many-to-one: many order lines point to ONE product row, so no line is duplicated or lost.
# Inner is safe here because every product_id in order_items exists in products (15 rows stay 15).
items_products = pd.merge(order_items, products, on="product_id")

In [32]:
check("2.3", lambda: len(items_products) == 15 and {"product_name", "price", "quantity"} <= set(items_products.columns))

Task 2.3: PASS


### Task 2.4 (4 pts): Keys with different names

In the table below the key is called `cust_id`. Merge `orders` with it using `left_on` and `right_on`, then drop the
redundant `cust_id` column. Store the result in `m`.

In [33]:
customers_renamed = customers.rename(columns={"customer_id": "cust_id"})

# The key has different names on each side, so name both. Both key columns are kept in the result,
# so we drop the redundant right-hand one (cust_id). Default how="inner" -> 11 rows (order 1009 has no customer).
m = pd.merge(orders, customers_renamed, left_on="customer_id", right_on="cust_id").drop(columns="cust_id")

In [34]:
check("2.4", lambda: len(m) == 11 and "cust_id" not in m.columns and "customer_id" in m.columns)

Task 2.4: PASS


### Task 2.5 (6 pts): Find the orphans with indicator

Do an **outer** merge of `orders` and `customers` on `customer_id` with `indicator=True` into `outer`.
Using the `_merge` column, create two sorted lists of integers:

* `orders_without_customer`: `order_id` of orders whose customer is **not** in `customers`
* `customers_without_orders`: `customer_id` of customers who **never ordered**

In [35]:
# indicator=True adds a "_merge" column: "both", "left_only" (only in orders) or "right_only" (only in customers).
outer = pd.merge(orders, customers, on="customer_id", how="outer", indicator=True)

# left_only rows = orders whose customer_id was not found in customers -> orphan orders.
orders_without_customer = sorted(outer.loc[outer["_merge"] == "left_only", "order_id"].astype(int))
# right_only rows = customers that matched no order. (Their order_id is NaN, so we take customer_id instead.)
customers_without_orders = sorted(outer.loc[outer["_merge"] == "right_only", "customer_id"].astype(int))

In [36]:
check("2.5", lambda: [int(x) for x in orders_without_customer] == [1009]
      and [int(x) for x in customers_without_orders] == [7, 8])

Task 2.5: PASS


In [37]:
# Given: the region of each city. The key of this table is its INDEX.
city_info = pd.DataFrame(
    {"region": ["South", "Capital", "South", "Central"]},
    index=["Almaty", "Astana", "Shymkent", "Karaganda"],
)
city_info

,region
Almaty,South
Astana,Capital
Shymkent,South
Karaganda,Central


### Task 2.6 (4 pts): Merge on the index and DataFrame.join

Add the `region` column to `customers` in two equivalent ways:

1. `with_region`: `pd.merge` with `left_on="city"` and `right_index=True`
2. `with_region2`: `customers.join(...)` with the `on` argument

In [38]:
# Left key is a COLUMN (city), right key is the INDEX of city_info -> left_on + right_index=True.
with_region = pd.merge(customers, city_info, left_on="city", right_index=True)

# DataFrame.join does the same: `on` names the calling frame's column, and it matches the other frame's INDEX.
# join defaults to how="left", so all 8 customers survive either way.
with_region2 = customers.join(city_info, on="city")

In [39]:
check("2.6", lambda: len(with_region) == 8 and "region" in with_region.columns
      and with_region.sort_index().equals(with_region2.sort_index()))

Task 2.6: PASS


---
# Part 3. Concatenation (14 points)

`pd.concat([a, b])` stacks objects. `ignore_index=True` renumbers the rows, `keys=[...]` labels the pieces,
`join="inner"` keeps only the shared labels of the other axis.

In [40]:
first_part = orders[orders["order_date"] < "2026-03-01"]    # Jan and Feb orders
second_part = orders[orders["order_date"] >= "2026-03-01"]  # Mar and Apr orders
print(len(first_part), len(second_part))

5 7


### Task 3.1 (5 pts): Stack rows

Put `first_part` and `second_part` back together into `all_orders` with a clean index `0..11`.

In [41]:
# ignore_index=True throws away the old labels (which would be 0..2,4.. gaps) and renumbers 0..11.
all_orders = pd.concat([first_part, second_part], ignore_index=True)

In [42]:
check("3.1", lambda: len(all_orders) == 12 and list(all_orders.index) == list(range(12)))

Task 3.1: PASS


### Task 3.2 (4 pts): keys

Concatenate the same two parts with the keys `"early"` and `"late"` into `by_part`.
Then select the `"late"` rows into `late_orders`.

In [43]:
# keys= adds an outer index level naming each piece -> a 2-level MultiIndex (part, original row label).
by_part = pd.concat([first_part, second_part], keys=["early", "late"])
# Selecting the outer label returns that piece (7 orders: March and April).
late_orders = by_part.loc["late"]

In [44]:
check("3.2", lambda: by_part.index.nlevels == 2 and len(late_orders) == 7)

Task 3.2: PASS


In [45]:
df_a = pd.DataFrame({"id": [1, 2, 3], "name": ["Ann", "Bob", "Cid"], "score": [88, 92, 75]})
df_b = pd.DataFrame({"id": [4, 5], "name": ["Dan", "Eva"], "grade": ["A", "B"]})

### Task 3.3 (5 pts): Different columns: outer vs inner

`df_a` and `df_b` share only `id` and `name`.

* `stacked_outer`: stack them keeping **all** columns (gaps become `NaN`), clean index
* `stacked_inner`: stack them keeping **only the shared** columns, clean index

In [46]:
# Default join="outer": union of columns; cells missing from a frame become NaN (score for Dan/Eva, grade for Ann/Bob/Cid).
stacked_outer = pd.concat([df_a, df_b], ignore_index=True)
# join="inner": keep only columns present in BOTH frames -> id and name.
stacked_inner = pd.concat([df_a, df_b], join="inner", ignore_index=True)

In [47]:
check("3.3", lambda: stacked_outer.shape == (5, 4) and stacked_inner.shape == (5, 2)
      and list(stacked_inner.columns) == ["id", "name"])

Task 3.3: PASS


---
# Part 4. `combine_first` (8 points)

`a.combine_first(b)` keeps the values of `a` and fills its **missing** values from `b`. The result covers both indexes.

In [48]:
supplier_a = pd.Series([450000, np.nan, 800, np.nan, 12000], index=[101, 102, 103, 104, 105], name="price")
supplier_b = pd.Series([24000, 750, 140, 9500], index=[102, 103, 104, 106], name="price")
print(supplier_a, "\n")
print(supplier_b)

101    450000.0
102         NaN
103       800.0
104         NaN
105     12000.0
Name: price, dtype: float64 

102    24000
103      750
104      140
106     9500
Name: price, dtype: int64


### Task 4.1 (8 pts): Best available price

Supplier A has priority. Fill A's missing prices from supplier B into `best_price`.
Product `103` is in both lists: A's price must win. Product `106` exists only at supplier B.

In [49]:
# combine_first: A's non-missing values win; NaNs in A are filled from B; the index is the UNION of both.
# 102, 104 are NaN in A -> filled from B; 103 exists in both -> A's 800 wins; 106 only in B -> appears (9500).
best_price = supplier_a.combine_first(supplier_b)

In [50]:
check("4.1", lambda: len(best_price) == 6 and best_price[102] == 24000 and best_price[103] == 800 and best_price[106] == 9500)

Task 4.1: PASS


---
# Part 5. Reshaping (22 points)

| From | To | Tool |
|---|---|---|
| long | wide | `df.pivot(index=..., columns=..., values=...)` |
| wide | long | `pd.melt(df, id_vars=..., var_name=..., value_name=...)` |
| inner row level | columns | `s.unstack()` |

**Long:** one row per observation. **Wide:** one column per category.

In [51]:
sales_long = pd.DataFrame({
    "month":    ["2026-01"] * 3 + ["2026-02"] * 3 + ["2026-03"] * 3,
    "category": ["Electronics", "Stationery", "Accessories"] * 3,
    "revenue":  [550, 9, 12, 25, 4, 12, 475, 40, 24],     # thousand KZT
})
sales_long

,month,category,revenue
0,2026-01,Electronics,550
1,2026-01,Stationery,9
2,2026-01,Accessories,12
3,2026-02,Electronics,25
4,2026-02,Stationery,4
5,2026-02,Accessories,12
6,2026-03,Electronics,475
7,2026-03,Stationery,40
8,2026-03,Accessories,24


### Task 5.1 (6 pts): pivot: long to wide

Make `wide`: **months as rows, categories as columns, revenue as values**. Use `pivot` with keyword arguments.

In [52]:
# pivot: long -> wide. Unique (month, category) pairs become cells; category values become columns (sorted alphabetically).
wide = sales_long.pivot(index="month", columns="category", values="revenue")

In [53]:
check("5.1", lambda: wide.shape == (3, 3) and list(wide.columns) == ["Accessories", "Electronics", "Stationery"]
      and wide.loc["2026-02", "Electronics"] == 25)

Task 5.1: PASS


### Task 5.2 (6 pts): melt: wide to long

Turn `wide` back into long format with `pd.melt` into `melted`. Use `wide.reset_index()`, keep `month` as the identifier,
and name the new columns `category` and `revenue`.

In [54]:
# melt needs a flat table, so reset_index() turns the "month" index back into a column.
# id_vars stay as-is; every other column (the categories) is stacked into rows:
# the old column NAMES go into var_name, the cell VALUES into value_name. 3 months x 3 categories = 9 rows.
melted = pd.melt(wide.reset_index(), id_vars="month", var_name="category", value_name="revenue")

In [55]:
check("5.2", lambda: len(melted) == 9 and list(melted.columns) == ["month", "category", "revenue"]
      and melted["revenue"].sum() == 1151)

Task 5.2: PASS


### Task 5.3 (5 pts): unstack

1. `multi`: take `sales_long`, make `month` and `category` the index and keep `revenue` as a Series.
2. `from_multi`: apply `unstack()` to `multi`. It should look exactly like `wide`.

In [56]:
# Selecting ["revenue"] after set_index gives a Series with a (month, category) MultiIndex.
multi = sales_long.set_index(["month", "category"])["revenue"]
# unstack() moves the INNER (last) index level into columns -> same table as pivot.
from_multi = multi.unstack()

In [57]:
check("5.3", lambda: from_multi.shape == (3, 3) and from_multi.loc["2026-03", "Electronics"] == 475
      and from_multi.equals(wide))

Task 5.3: PASS


### Task 5.4 (5 pts): Round trip

Reshape `melted` (long) to wide with `pivot` into `back`. It must be identical to `wide`.

In [58]:
# Round trip: melt (wide->long) is undone by pivot (long->wide) with the same three column names.
back = melted.pivot(index="month", columns="category", values="revenue")

In [59]:
check("5.4", lambda: back.equals(wide))

Task 5.4: PASS


---
# Part 6. Mini project: revenue report (18 points)

The manager wants a report on **paid orders**. Revenue of an order line is `quantity * price`.

### Task 6.1 (7 pts): Build one big table

Build `full`:

1. Start with `order_items` and merge `products` on `product_id`.
2. Merge `orders` on `order_id`.
3. Merge `customers` on `customer_id` with a **left** join (keep orders of unknown customers) and
   `suffixes=("_order", "_customer")`, because `status` exists in both tables.
4. Add `revenue = quantity * price`.
5. Keep only paid orders (`status_order == "paid"`) and call `.copy()`.

In [60]:
# 1) every order line + its product (inner is fine: all product_ids exist)
full = pd.merge(order_items, products, on="product_id")
# 2) + order header (order_id, customer_id, order_date, status)
full = pd.merge(full, orders, on="order_id")
# 3) + customer. LEFT join so order 1009 (customer 99, unknown) is NOT dropped.
#    "status" exists in orders and customers, so suffixes make status_order / status_customer.
full = pd.merge(full, customers, on="customer_id", how="left", suffixes=("_order", "_customer"))
# 4) revenue of each line
full["revenue"] = full["quantity"] * full["price"]
# 5) only paid orders (drops cancelled order 1004). .copy() avoids SettingWithCopy warnings later.
full = full[full["status_order"] == "paid"].copy()

In [61]:
check("6.1", lambda: len(full) == 14 and full["revenue"].sum() == 1576900 and (full["status_order"] == "paid").all())

Task 6.1: PASS


### Task 6.2 (6 pts): Revenue by city and category

1. `full_clean`: a copy of `full` where missing `city` and `name` are replaced by `"Unknown"`.
2. `report`: **cities as rows, categories as columns**, summing `revenue`, empty cells `0`.

*Hint:* `fillna` and `pivot_table(index=..., columns=..., values=..., aggfunc="sum", fill_value=0)`.

In [62]:
# The unknown customer (99) gave NaN city/name from the left join; label them "Unknown" so they appear in the report.
full_clean = full.copy()
full_clean[["city", "name"]] = full_clean[["city", "name"]].fillna("Unknown")

# pivot_table (unlike pivot) can AGGREGATE duplicates: several lines share a city+category, so we sum revenue.
# fill_value=0 replaces empty city/category combinations.
report = full_clean.pivot_table(index="city", columns="category", values="revenue", aggfunc="sum", fill_value=0)

In [63]:
check("6.2", lambda: full_clean["city"].isna().sum() == 0 and report.shape == (5, 3) and "Unknown" in report.index
      and report.to_numpy().sum() == 1576900 and report.loc["Unknown", "Electronics"] == 450000)

Task 6.2: PASS


### Task 6.3 (5 pts): Best customers

`top_customers`: total revenue per customer **name** from `full_clean`, sorted from largest to smallest.

In [64]:
# group by customer name, sum revenue, sort descending (largest first).
top_customers = full_clean.groupby("name")["revenue"].sum().sort_values(ascending=False)

In [65]:
check("6.3", lambda: top_customers.index[0] == "Aigerim" and top_customers.iloc[0] == 587000 and top_customers.sum() == 1576900)

Task 6.3: PASS


---
# Final score
Run the next cell.

In [66]:
summary()

,max,earned,result
task,,,
1.1,3,3,PASS
1.2,3,3,PASS
1.3,3,3,PASS
1.4,3,3,PASS
2.1,3,3,PASS
2.2,4,4,PASS
2.3,5,5,PASS
2.4,4,4,PASS
2.5,6,6,PASS



Part 1:  12 / 12
Part 2:  26 / 26
Part 3:  14 / 14
Part 4:   8 / 8
Part 5:  22 / 22
Part 6:  18 / 18
TOTAL (automatic): 100 / 100


---
# Defense questions

Prepare short oral answers. The instructor may ask any of these and may ask you to change your code live.

1. Explain the difference between `inner`, `left`, `right` and `outer`. Which one did you use in task 6.1 and why?
2. Why does the inner merge of `orders` and `customers` return 11 rows and not 12? Which order disappears?
3. When do you use `merge` and when `concat`?
4. What does `unstack` do? What is its inverse? What is the difference between `pivot` and `melt`?
5. What is a `MultiIndex` and how do you select the inner level of a Series?
6. What does `combine_first` do and which values win?
7. **Live demo:** in task 6.1 change `how="left"` to `how="inner"`. What happens to the number of rows and to the total revenue? Why?